# 02 — Temel NLP Görevleri

**Kapsam:** Metin sınıflandırma, varlık tanıma, özetleme, soru-cevap, diyalog yönetimi
ve makine çevirisi gibi NLP görevleri için uçtan uca çözümler.

Bu notebook, `src/nlp_tasks/` altındaki her modülü tek tek çalıştırıp çıktısını inceler.

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) unpack adımını atlar.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
#   4) EN ÖNEMLİSİ: data/, models/, mlruns/ klasörlerini Drive'daki kalıcı bir
#      klasöre sembolik bağlantı (symlink) yapar. Neden gerekli: /content her
#      runtime'da sıfırlanır, yani 01. notebook'ta ürettiğiniz corpus.jsonl gibi
#      dosyalar farklı bir runtime'da (örn. 02. notebook'u açtığınızda) KAYBOLUR.
#      Bu adım olmadan her notebook'u ayrı ayrı çalıştırdığınızda önceki adımların
#      ürettiği veriyi bulamazsınız. Sembolik bağlantı sayesinde hangi runtime'da
#      olursanız olun aynı kalıcı depoyu okur/yazarsınız.
import os, sys
os.environ.setdefault("USE_TF", "0")  # transformers TensorFlow'u hic denemesin (Colab'da protobuf catismasi yasatiyor)

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"
DRIVE_DATA_DIR = "/content/drive/MyDrive/baykar-nlp-hazirlik-data"
PERSIST_DIRS = ["data/raw", "data/processed", "models", "mlruns"]  # chroma_db BILEREK haric (asagida)

try:
    from google.colab import drive
    # force_remount=True KULLANMIYORUZ: bu, zaten mount edilmişken bile her seferinde
    # yeniden yetkilendirme (izin penceresi) ister, gereksiz bekleme/kesinti yaratır.
    # drive.mount() zaten mount edilmişse kendi içinde anında geri döner; mount
    # edilmemişse (bu runtime'da ilk çalıştırma) normal şekilde izin ister — bu
    # durumda çıkan izin penceresini/bağlantısını tamamlamanız gerekir, hücreyi
    # durdurmayın.
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False  # Colab dışında (yerelde) çalışıyorsanız Drive adımları atlanır.

if not os.path.exists(PROJECT_DIR) and IN_COLAB:
    if os.path.exists(DRIVE_ZIP_PATH):
        import shutil
        shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
    else:
        print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
              "yükleyin ya da kendi reponuzu klonlayın: "
              f"!git clone <repo-url> {PROJECT_DIR}")

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

if IN_COLAB and os.path.exists(PROJECT_DIR):
    import shutil
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    for _name in PERSIST_DIRS:
        _drive_path = os.path.join(DRIVE_DATA_DIR, _name)
        os.makedirs(_drive_path, exist_ok=True)
        _local_path = os.path.join(PROJECT_DIR, _name)
        os.makedirs(os.path.dirname(_local_path), exist_ok=True)  # orn. data/ klasorunu gercek dizin olarak olustur

        if os.path.islink(_local_path):
            continue  # zaten Drive'a bağlanmış

        if os.path.isdir(_local_path):
            # Zip'ten gelen boş klasörü kaldırıp yerine symlink koyuyoruz. İçinde
            # (nadiren) veri varsa önce Drive'a taşıyoruz, hiçbir şeyi kaybetmiyoruz.
            for _item in os.listdir(_local_path):
                _src = os.path.join(_local_path, _item)
                _dst = os.path.join(_drive_path, _item)
                if not os.path.exists(_dst):
                    shutil.move(_src, _dst)
            shutil.rmtree(_local_path)

        os.symlink(_drive_path, _local_path)

    print("Kalıcı veri klasörü:", DRIVE_DATA_DIR)
    print("Not: chroma_db (vektor veritabani) Drive'a BAGLANMADI -- SQLite, Drive'in")
    print("     FUSE dosya sisteminde yazma kilidini desteklemiyor ('OperationalError:")
    print("     attempt to write a readonly database'). Her yeni runtime'da RAG")
    print("     notebook'undaki (03) indeksleme hucresini tekrar calistirin -- chunks.jsonl")
    print("     zaten Drive'da oldugu icin bu hizli ve ucretsiz bir islemdir.")


## Colab ortam düzeltmeleri

Kurulum hücresinden hemen sonra çalıştırın. Transformers v5 pipeline yamaları ve paket sürümleri.

In [ ]:
import os, subprocess, sys

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
bootstrap = os.path.join(PROJECT_DIR, "scripts", "colab_bootstrap.py")

if not os.path.exists(bootstrap):
    raise FileNotFoundError(
        "scripts/colab_bootstrap.py bulunamadi. "
        "Guncel projeyi zip'leyip Drive'a yukleyin."
    )

subprocess.run([sys.executable, bootstrap], check=True, cwd=PROJECT_DIR)

### ⚠️ Kurulum sonrası zorunlu: Runtime yeniden başlatma

`colab_bootstrap.py` numpy/scipy dahil birçok paketi force-reinstall ediyor. Colab'ın
kernel'i numpy'yi bu hücre çalışmadan ÖNCE zaten bellekte tutuyor olabileceği için,
yeniden başlatmadan devam ederseniz sonraki hücrelerde
`ImportError: cannot import name '_center' from numpy._core.umath` hatası alabilirsiniz.

1. **Runtime → Oturumu yeniden başlat** (yukarıdaki kurulum hücresini TEKRAR ÇALIŞTIRMAYIN)
2. Bu notebook'un en üstteki (Drive/symlink) hücresini tekrar çalıştırın
3. Aşağıdaki doğrulama hücresini çalıştırıp devam edin

In [ ]:
import numpy as np
from numpy._core import strings  # eskiden "_center" ImportError'i buradan geliyordu
from transformers.models.qwen2 import modeling_qwen2  # bu projenin gercek modeli -- scipy/numpy ABI sorunlari (orn. "_blas_supports_fpe") transformers'in object-detection loss modulu uzerinden burada ortaya cikardi
print("numpy:", np.__version__, "OK -- devam edebilirsiniz.")

## Sınıflandırma (zero-shot)

Etiketli veri toplamadan önce hızlı bir başlangıç noktası: doküman türünü tahmin eder.

In [ ]:
from src.nlp_tasks.classification import zero_shot_classify

text = "Bu bölümde sistemin ana bileşenleri ve bunlar arasındaki veri akışı açıklanmaktadır."
result = zero_shot_classify(text)
print(result["label"])
print(result["scores"])


## Sınıflandırma (fine-tuned)

Zero-shot pratik bir başlangıç ama gerçek bir eğitim değil. Burada `dbmdz/bert-base-turkish-cased`
üzerine gerçek bir ince ayar (fine-tuning) yapıyoruz: elle etiketlemeye gerek yok — her chunk,
kaynak şablonun başlığından (`doc_title`) miras aldığı türü zaten biliyor, biz sadece bunu
`CLASSIFICATION_LABELS`'a eşliyoruz. 06. notebook'taki budama (pruning) demosu bu sınıflandırıcıyı
kullanıyor; bir kere çalıştırmanız yeterli (sonuç Drive'a kalıcı, `models/classifier`).

In [ ]:
import json, random
from src.config import CLASSIFICATION_LABELS
from src.nlp_tasks.classification import train

def _label_for_title(doc_title: str) -> str:
    base = doc_title.replace(" Şablonu", "").split(" (")[0].strip()
    return base if base in CLASSIFICATION_LABELS else "Diğer"

with open("data/processed/chunks.jsonl", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f]

pairs = [(c["text"], CLASSIFICATION_LABELS.index(_label_for_title(c["doc_title"]))) for c in chunks]
random.Random(42).shuffle(pairs)  # train() sirali 85/15 boluyor; karistirmazsak eval seti son sablonlarla sinirli kalirdi
texts, labels = zip(*pairs)

classifier_dir = train(list(texts), list(labels))
print(f"Sınıflandırıcı kaydedildi -> {classifier_dir}")

## Varlık Tanıma (NER)

Genel model + teknik-dokümantasyona özgü regex desenleri (kod parçası, CLI bayrağı, sürüm, dosya yolu) birleşimi.

In [ ]:
from src.nlp_tasks.ner import extract_entities

sample = "`kullanici_olustur(ad, e_posta)` fonksiyonunu çağırmadan önce DATABASE_URL ortam değişkenini v2.3.0 sürümünde /etc/proje/config.yaml içinde ayarlayın."
for ent in extract_entities(sample):
    print(ent)


## Özetleme

Extractive (hızlı, halüsinasyonsuz) vs. abstractive (akıcı) karşılaştırması.

In [ ]:
from src.nlp_tasks.summarization import summarize

with open("data/raw/corpus.jsonl", encoding="utf-8") as f:
    import json
    long_text = json.loads(f.readline())["text"]

print("EXTRACTIVE:\n", summarize(long_text, method="extractive", sentence_count=3))
print("\nABSTRACTIVE:\n", summarize(long_text, method="abstractive"))


## Soru-Cevap (extractive)

In [ ]:
from src.nlp_tasks.qa import extractive_answer

ctx = "kullanici_olustur fonksiyonu, e_posta parametresi zaten kayıtlıysa DuplicateEmailError fırlatır."
print(extractive_answer("kullanici_olustur hangi hatayı fırlatır?", ctx))


## Makine Çevirisi

In [ ]:
from src.nlp_tasks.translation import translate

print(translate("Bu fonksiyon, kullanıcı e-postası zaten kayıtlıysa bir hata fırlatır.", "tr-en"))


## Diyalog Yönetimi

`dialogue.py`, RAG pipeline'ına (03. notebook'ta kuracağız) bağımlıdır — vektör veritabanı
indekslenmeden bu hücre çalışmaz. Şimdilik sadece bağlam yeniden yazma (query rewriting)
mekanizmasını inceleyelim.

In [ ]:
from src.nlp_tasks.dialogue import DialogueSession, rewrite_standalone_query

session = DialogueSession(session_id="demo")
session.add_user_turn("İyi bir README nasıl yapılandırılır?")
session.add_assistant_turn("README; özellikler, kurulum, kullanım ve lisans bölümlerini içermelidir.")

standalone = rewrite_standalone_query(session, "Peki kurulum bölümüne ne yazmalıyım?")
print(standalone)
